# Hacash CUDA smoke (Colab T4)

**Phase 1 only:** build + unit tests for `x16rs-cuda` and `poworker --features cuda`.

- Does **not** sync mainnet.
- Does **not** push to GitHub.
- **Runtime → Change runtime type → T4 GPU** before running.

## PASS criteria
1. `nvidia-smi` shows a GPU (T4 is fine, sm_75).
2. `cargo test -p x16rs-cuda --features cuda` passes (genesis + CPU/GPU match + batch).
3. `cargo build --release --bin poworker --features cuda` succeeds.
4. Log files under `scripts/mining-nvidia/colab-results/`.

## 1) GPU check

In [ ]:
!nvidia-smi

## 2) Get the repo

Option A: clone your public fork (edit URL if needed).

Option B: upload a zip of the repo to Colab and unzip, then set `REPO_DIR`.

In [ ]:
import os
from pathlib import Path

# --- edit if needed ---
REPO_URL = "https://github.com/Moskyera/fullnodedev.git"
BRANCH = "main"  # or your working branch with x16rs-cuda
REPO_DIR = Path("/content/fullnodedev")

if not REPO_DIR.exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo already present:", REPO_DIR)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
assert (REPO_DIR / "x16rs-cuda").is_dir(), "x16rs-cuda missing — wrong branch or incomplete tree"
assert (REPO_DIR / "scripts/mining-nvidia/colab_cuda_smoke.sh").is_file(), "smoke script missing — pull latest Phase 1 files"

## 3) Run smoke script (FREE TIER)

Default = **only CUDA crate tests** (~10–25 min). Not full poworker release.

You should see `still working...` every minute. If silent for 10+ min, interrupt and re-run.

If you still have the **old** never-ending build: **Runtime → Interrupt execution**, then run this cell.

In [ ]:
import os
from pathlib import Path

# Prefer slim unpack path; fall back to clone path
for d in ("/content/hacash-fullnodedev", "/content/fullnodedev"):
    if Path(d).is_dir():
        os.chdir(d)
        break
print("cwd:", Path.cwd())

!chmod +x scripts/mining-nvidia/colab_cuda_smoke.sh
# FREE tier: do NOT set COLAB_FULL=1
!bash scripts/mining-nvidia/colab_cuda_smoke.sh

summary = Path("scripts/mining-nvidia/colab-results/latest-summary.txt")
if summary.is_file():
    print("--- latest-summary.txt ---")
    print(summary.read_text())
else:
    print("No summary file — script failed early")

## 4) Download evidence (optional)

After PASS, download logs from the left file browser:
`fullnodedev/scripts/mining-nvidia/colab-results/`

Keep them offline. **Do not publish a production CUDA claim without these logs.**

In [ ]:
from pathlib import Path
from google.colab import files

results = Path("/content/fullnodedev/scripts/mining-nvidia/colab-results")
if results.is_dir():
    for p in sorted(results.glob("*")):
        print("download:", p)
        files.download(str(p))
else:
    print("No results dir yet")